In [3]:
import sys
sys.path.append('..')
import requests
import os
import random
from helpers import *
from dotenv import load_dotenv

In [4]:
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

In [5]:
AVAILABLE_PLATFORMS = [
    "Netflix",
    "Amazon Prime Video",
    "JioHotstar",
    "Sony Liv",
    "Zee5",
    "YouTube"
]

In [6]:
LANGUAGE_CODES = {
    "Any": None,
    "English": "en",
    "Hindi": "hi",
    "Kannada": "kn",
    "Telugu": "te",
    "Tamil": "ta",
    "Malayalam": "ml",
    "Korean": "ko",
}

In [7]:
def get_watch_providers(title, region="IN"):
    """
    Get streaming platforms for a movie in a region.
    Returns a list of platform names.
    """
    try:
        # Step A — get the movie's TMDB ID first
        clean_title = title.split("(")[0].strip()
        year = title.split("(")[1].replace(")", "").strip()
        
        search_url = "https://api.themoviedb.org/3/search/movie"
        params = {
            "api_key": TMDB_API_KEY,
            "query": clean_title,
            "year": year
        }
        response = requests.get(search_url, params=params, timeout=5)
        data = response.json()
        
        if not data["results"]:
            return []
        
        movie_id = data["results"][0]["id"]   # get the ID
        
        # Step B — call the watch providers endpoint
        provider_url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers"
        
        # Step B — call the watch providers endpoint
        prov_params = {"api_key": TMDB_API_KEY}
        
        prov_response = requests.get(provider_url, params=prov_params, timeout=5)
        prov_data = prov_response.json()
        
        # Step C — dig into results → region → flatrate
        results = prov_data.get("results", {})
        region_data = results.get(region, {})
        flatrate = region_data.get("flatrate", [])
        
        # Step D — extract just the platform names
        platforms = [p["provider_name"] for p in flatrate]
        
        return platforms
        
    except:
        return []

In [8]:
# Test with a popular movie
platforms = get_watch_providers("Toy Story (1995)")
print("Toy Story is on:", platforms)

Toy Story is on: ['JioHotstar', 'VI movies and tv']


In [9]:
def filter_by_platform(titles, user_platforms, region="IN"):
    """
    Keep only movies available on the user's platforms.
    Uses 'contains' matching to catch variations.
    """
    filtered = []
    
    for title in titles:
        movie_platforms = get_watch_providers(title, region)
        
        # Check if ANY user platform matches (contains)
        for user_plat in user_platforms:
            for movie_plat in movie_platforms:
                if user_plat in movie_plat:   #  'contains' check!
                    filtered.append(title)
                    break
            else:
                continue   # no match, keep checking
            break   # match found, stop
    
    return filtered

In [10]:
movies = ["Toy Story (1995)", "Jumanji (1995)", "The Dark Knight (2008)"]
user_platforms = ["Amazon Prime Video"]

result = filter_by_platform(movies, user_platforms)
print("On Amazon Prime Video:")
for m in result:
    print("  -", m)

On Amazon Prime Video:
  - Jumanji (1995)
  - The Dark Knight (2008)


In [11]:
jumanji_platforms = get_watch_providers("Jumanji (1995)")
print("Jumanji is on:", jumanji_platforms)

Jumanji is on: ['Amazon Prime Video', 'Lionsgate Play', 'Lionsgate Play Apple TV Channel', 'Lionsgate Play Amazon Channel', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']


In [12]:

def get_recommendations_with_platform(genre, user_platforms=None, n=5, user_id=None, region="IN"):
    """
    Get recommendations, optionally filter by platform.
    If user_platforms is empty → skip filtering.
    """
    # If no platforms chosen → skip filter
    if not user_platforms:
        return get_recommendations(genre=genre, n=n, user_id=user_id)
    
    # Otherwise → buffer, filter, return top n
    buffer_n = n * 4
    titles = get_recommendations(genre=genre, n=buffer_n, user_id=user_id)
    filtered = filter_by_platform(titles, user_platforms, region)
    return filtered[:n]

In [13]:
result = get_recommendations_with_platform(
    genre="Comedy",
    user_platforms=["JioHotstar", "Netflix"],
    n=5
)
print("Comedy movies you can watch:")
for movie in result:
    print("  →", movie)

Comedy movies you can watch:


In [14]:
# Step 1 — get the buffer (before filtering)
buffer_movies = get_recommendations(genre="Comedy", n=20, user_id=None)
print("Got", len(buffer_movies), "movies before filtering")

# Step 2 — check how many survive the filter
filtered = filter_by_platform(buffer_movies, ["JioHotstar", "Netflix"])
print("After filtering:", len(filtered), "movies survived")
print()

# Step 3 — see which survived
for m in filtered:
    print("  →", m)

Got 10 movies before filtering
After filtering: 1 movies survived

  → Kiss Kiss Bang Bang (2005)


In [15]:
# Check what TMDB calls each platform
test_movies = [
    "Toy Story (1995)",
    "Jumanji (1995)",
    "The Dark Knight (2008)",
    "Inception (2010)",
]

for movie in test_movies:
    platforms = get_watch_providers(movie)
    print(f"{movie}:")
    print(f"  {platforms}\n")

Toy Story (1995):
  ['JioHotstar', 'VI movies and tv']

Jumanji (1995):
  ['Amazon Prime Video', 'Lionsgate Play', 'Lionsgate Play Apple TV Channel', 'Lionsgate Play Amazon Channel', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']

The Dark Knight (2008):
  ['Amazon Prime Video', 'JioHotstar', 'Amazon Prime Video with Ads']

Inception (2010):
  ['Amazon Prime Video', 'JioHotstar', 'Amazon Prime Video with Ads']



In [16]:


# Quick test
platforms = get_watch_providers("Toy Story (1995)")
print("Works from helpers:", platforms)

Works from helpers: ['JioHotstar', 'VI movies and tv']


In [17]:

buffer = get_recommendations(genre="Animation", n=24, user_id=1)
print("Movies BEFORE filter:", len(buffer))

# Filter by Netflix
filtered = filter_by_platform(buffer, ["Netflix"])
print("Movies AFTER Netflix filter:", len(filtered))
print()
for m in filtered:
    print("  →", m)

Movies BEFORE filter: 20
Movies AFTER Netflix filter: 3

  → Princess Mononoke (Mononoke-hime) (1997)
  → Spirited Away (Sen to Chihiro no kamikakushi) (2001)
  → Howl's Moving Castle (Hauru no ugoku shiro) (2004)


In [18]:


# Get Animation movies (like your test)
buffer = get_recommendations(genre="Animation", n=24, user_id=1)
print("Movies BEFORE filter:", len(buffer))

# Filter by Netflix
filtered = filter_by_platform(buffer, ["Netflix"])
print("Movies AFTER Netflix filter:", len(filtered))
print()
for m in filtered:
    print("  →", m)

Movies BEFORE filter: 20
Movies AFTER Netflix filter: 4

  → Laputa: Castle in the Sky (Tenkû no shiro Rapyuta) (1986)
  → Princess Mononoke (Mononoke-hime) (1997)
  → Spirited Away (Sen to Chihiro no kamikakushi) (2001)
  → Howl's Moving Castle (Hauru no ugoku shiro) (2004)


In [19]:
for movie in ["Toy Story (1995)", "Shrek (2001)"]:
    print(movie, get_watch_providers(movie))

Toy Story (1995) ['JioHotstar', 'VI movies and tv']
Shrek (2001) ['Amazon Prime Video', 'Amazon Prime Video with Ads']


In [20]:

def get_genre_map():
    """
    Build a dictionary that translates genre NAMES → TMDB genre IDs.
    e.g. {"Animation": 16, "Action": 28, ...}
    """
    url = "https://api.themoviedb.org/3/genre/movie/list"
    params = {"api_key": TMDB_API_KEY, "language": "en-US"}
    response = requests.get(url, params=params, timeout=5).json()

    genre_map = {}
    for g in response["genres"]:
        genre_map[g["name"]] = g["id"]     # "Animation" → 16

    return genre_map

In [21]:
GENRE_MAP = get_genre_map()
print(GENRE_MAP)

{'Action': 28, 'Adventure': 12, 'Animation': 16, 'Comedy': 35, 'Crime': 80, 'Documentary': 99, 'Drama': 18, 'Family': 10751, 'Fantasy': 14, 'History': 36, 'Horror': 27, 'Music': 10402, 'Mystery': 9648, 'Romance': 10749, 'Science Fiction': 878, 'TV Movie': 10770, 'Thriller': 53, 'War': 10752, 'Western': 37}


In [22]:

LLM_TO_TMDB_GENRE = {
    "Sci-Fi":   "Science Fiction",
    "Children": "Family",
    "Musical":  "Music",
}

def get_genre_id(llm_genre):
    tmdb_name = LLM_TO_TMDB_GENRE.get(llm_genre, llm_genre)   # translate, or keep as-is
    drama_id  = GENRE_MAP["Drama"]
    genre_id  = GENRE_MAP.get(tmdb_name, drama_id)            # find ID, or fall back to Drama
    return genre_id

In [23]:
print(get_genre_id("Action"))      # expect 28  (passes straight through)
print(get_genre_id("Sci-Fi"))      # expect 878 (translated → Science Fiction)
print(get_genre_id("Film-Noir"))   # expect 18  (unknown → falls back to Drama)
print(get_genre_id("Comedy"))      # expect 35  (passes straight through)

28
878
18
35


In [24]:
PLATFORM_IDS = {
    "Netflix": 8,
    "Amazon Prime Video": 119,
    "JioHotstar": 2336,
    "Zee5": 232,
    "Sony Liv": 237,
    "YouTube": 192,
}

In [25]:
def discover_movies_by_genre(genre_id, n=20, provider_ids=None, language=None):
    url = "https://api.themoviedb.org/3/discover/movie"
    is_regional = language in ["kn", "te", "ml", "ta"]

    params = {
        "api_key": TMDB_API_KEY,
        "with_genres": genre_id,
        "sort_by": "popularity.desc",
        "watch_region": "IN",
    }
    if not is_regional:
        params["vote_count.gte"] = 100     # standard gate for global content
    if provider_ids:
        params["with_watch_providers"] = "|".join(str(pid) for pid in provider_ids)
        params["watch_region"] = "IN"
        params["with_watch_monetization_types"] = "flatrate"
    if language:
        params["with_original_language"] = language

    # Step 1: fetch page 1 to learn how many pages actually exist
    first = requests.get(url, params={**params, "page": 1}, timeout=5).json()
    total_pages = first.get("total_pages", 1)

    # Step 2: pick a random page within the REAL range (capped at 10 to avoid
    # drifting into very obscure, low-popularity results even for regional content)
    max_page = min(total_pages, 10)
    page = random.randint(1, max_page)

    if page == 1:
        return first.get("results", [])[:n]      # reuse — no extra call needed
    
    response = requests.get(url, params={**params, "page": page}, timeout=5).json()
    return response.get("results", [])[:n]

In [26]:
movies = discover_movies_by_genre(878, n=5)   # 878 = Sci-Fi

print("How many movies:", len(movies))
print()

# Peek at the FIRST movie's shape
first = movies[0]
print("Title:", first["title"])
print("ID:", first["id"])
print("Overview:", first["overview"][:80], "...")
print("Poster:", first["poster_path"])

How many movies: 5

Title: Mercy
ID: 1236153
Overview: In the near future, a detective stands on trial accused of murdering his wife. H ...
Poster: /pyok1kZJCfyuFapYXzHcy7BLlQa.jpg


In [27]:
def get_providers_by_id(movie_id, region="IN"):
    try:
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers"
        params = {"api_key": TMDB_API_KEY}
        response = requests.get(url, params=params, timeout=10).json()
        results = response.get("results", {})
        region_data = results.get(region, {})
        flatrate = region_data.get("flatrate", [])
        return [p["provider_name"] for p in flatrate]
    except:
        return []

In [28]:
# Get fresh sci-fi movies, then check the FIRST one's platforms
movies = discover_movies_by_genre(878, n=5)

first = movies[0]
print("Movie:", first["title"])
print("ID:", first["id"])

platforms = get_providers_by_id(first["id"])
print("Platforms:", platforms)

Movie: 28 Days Later
ID: 170
Platforms: ['Amazon Prime Video', 'Amazon Prime Video with Ads']


In [29]:
movies = discover_movies_by_genre(878, n=5)

for m in movies:
    platforms = get_providers_by_id(m["id"])
    print(m["title"], "→", platforms)

War Machine → ['Netflix']
Predator: Badlands → ['JioHotstar']
The Substance → ['MUBI', 'MUBI Amazon Channel']
Dune → ['Netflix', 'Amazon Prime Video', 'Amazon Prime Video with Ads']
Spider-Man 2 → ['Netflix', 'Amazon Prime Video', 'JioHotstar', 'Sony Liv', 'Lionsgate Play', 'Lionsgate Play Apple TV Channel', 'Lionsgate Play Amazon Channel', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']


In [30]:
def guest_recommendations_with_platform(genre_name, user_platforms=None, n=5, region="IN",language=None):
    genre_id = get_genre_id(genre_name)
    

    # NEW — convert platform names → provider IDs
    provider_ids = None
    if user_platforms:
        provider_ids = [PLATFORM_IDS[p] for p in user_platforms if p in PLATFORM_IDS]

    lang_code = LANGUAGE_CODES.get(language)   # None if not found or not given
    
    # Pass provider_ids into discover
    buffer = discover_movies_by_genre(genre_id, n=n * 4, provider_ids=provider_ids, language=lang_code)

    

    survivors = []
    for movie in buffer:
        platforms = get_providers_by_id(movie["id"], region)
        if not platforms:
            continue
        if not user_platforms:
            movie["platforms"] = platforms
            survivors.append(movie)
        else:
            for user_plat in user_platforms:
                for movie_plat in platforms:
                    if user_plat.lower() in movie_plat.lower():
                        movie["platforms"] = platforms
                        survivors.append(movie)
                        break
                else:
                    continue
                break
        if len(survivors) >= n:
            break
    return survivors

In [31]:
guest_recommendations_with_platform(genre_name="Crime", language="Korean", n=6)

[{'adult': False,
  'backdrop_path': '/1eM6Ey1FzpgJigrxRiSYtntSWq7.jpg',
  'genre_ids': [35, 80, 53],
  'id': 639988,
  'title': 'No Other Choice',
  'original_language': 'ko',
  'original_title': '어쩔수가없다',
  'overview': 'After being laid off and humiliated by a ruthless job market, a veteran paper mill manager descends into violence in a desperate bid to reclaim his dignity.',
  'popularity': 18.5554,
  'poster_path': '/vc2S0dvgpsM0XfSiXZDMVkRCSSU.jpg',
  'release_date': '2025-09-24',
  'softcore': False,
  'video': False,
  'vote_average': 7.499,
  'vote_count': 1211,
  'platforms': ['MUBI', 'MUBI Amazon Channel']},
 {'adult': False,
  'backdrop_path': '/mIW7eFxHnNqlwKwKWb6dpVjmHao.jpg',
  'genre_ids': [28, 53, 27, 80],
  'id': 799379,
  'title': 'Project Wolf Hunting',
  'original_language': 'ko',
  'original_title': '늑대사냥',
  'overview': 'While under heavily armed guard, the dangerous convicts aboard a cargo ship unite in a coordinated escape attempt that soon escalates into a bloo

In [32]:
print(LANGUAGE_CODES.get("Korean"))    # should print "ko"
print(LANGUAGE_CODES.get("korean"))    # will print None — case matters

ko
None


In [33]:
import inspect
print(inspect.signature(guest_recommendations_with_platform))

(genre_name, user_platforms=None, n=5, region='IN', language=None)


In [34]:
results = guest_recommendations_with_platform(genre_name="Crime", language="Korean", n=6)
for m in results:
    print(m["title"], "|", m.get("original_language"))

Nameless Gangster | ko
Door Lock | ko
The Chase | ko
Hwayi: A Monster Boy | ko
Hard Hit | ko
Bogotá: City of the Lost | ko


In [35]:
test = discover_movies_by_genre(get_genre_id("Crime"), n=6, language="ko")
for m in test:
    print(m["title"], "|", m.get("original_language"))

The Divine Move | ko
Breath | ko
Friend | ko
The City of Violence | ko
The Swindlers | ko
Master | ko


In [36]:
# Test A — a genre that goes through TRANSLATION (Children → Family)
results = guest_recommendations_with_platform(
    genre_name="Children",
    user_platforms=["Netflix", "Amazon Prime Video", "JioHotstar"],
    n=5
)
print("CHILDREN →", len(results), "found")
for m in results:
    print(" ", m["title"], "→", m["platforms"])

print()

# Test B — a normal pass-through genre, streaming-poorer maybe (Horror)
results = guest_recommendations_with_platform(
    genre_name="Horror",
    user_platforms=["Netflix", "Amazon Prime Video", "JioHotstar"],
    n=5
)
print("HORROR →", len(results), "found")
for m in results:
    print(" ", m["title"], "→", m["platforms"])

CHILDREN → 5 found
  Freaky Friday → ['JioHotstar']
  The Princess Diaries → ['JioHotstar']
  Mrs. Doubtfire → ['JioHotstar', 'VI movies and tv']
  Monsters University → ['JioHotstar']
  Percy Jackson & the Olympians: The Lightning Thief → ['JioHotstar', 'VI movies and tv']

HORROR → 5 found
  Jaws 2 → ['JioHotstar']
  Incantation → ['Netflix']
  Locked → ['Amazon Prime Video', 'Lionsgate Play', 'Lionsgate Play Apple TV Channel', 'Lionsgate Play Amazon Channel', 'Amazon Prime Video with Ads']
  #Alive → ['Netflix']
  Marrowbone → ['Amazon Prime Video', 'Amazon Prime Video with Ads']


In [37]:
# Check what TMDB actually calls these platforms
movies = discover_movies_by_genre(get_genre_id("Drama"), n=24)
all_platforms = set()
for m in movies:
    for p in get_providers_by_id(m["id"]):
        all_platforms.add(p)

print(sorted(all_platforms))

['Amazon Prime Video', 'Amazon Prime Video with Ads', 'Apple TV', 'Apple TV Amazon Channel', 'JioHotstar', 'MGM Plus Amazon Channel', 'Netflix', 'Sony Liv', 'VI movies and tv']


In [38]:
all_platforms = set()

for g in ["Drama", "Animation", "Action", "Comedy", "Romance", "Thriller"]:
    movies = discover_movies_by_genre(get_genre_id(g), n=20)
    for m in movies:
        for p in get_providers_by_id(m["id"]):
            all_platforms.add(p)

print(sorted(all_platforms))

['Amazon Prime Video', 'Amazon Prime Video with Ads', 'Apple TV', 'Apple TV Amazon Channel', 'Crunchyroll', 'Crunchyroll Amazon Channel', 'FilmBox+', 'JioHotstar', 'Lionsgate Play', 'Lionsgate Play Amazon Channel', 'Lionsgate Play Apple TV Channel', 'Lionsgate+ Amazon Channels', 'Netflix', 'Sony Liv', 'Sony Pictures Amazon Channel', 'VI movies and tv']


In [39]:
movies = discover_movies_by_genre(get_genre_id("Drama"), n=10, provider_ids=[232])
print("Count:", len(movies))
for m in movies:
    print(m["title"])

Count: 10
Hotel Mumbai
BlackBerry
Steve Jobs
Contraband
Are You There God? It's Me, Margaret.
Extremely Wicked, Shockingly Evil and Vile
Mothering Sunday
Bob Marley: One Love
Mary Queen of Scots
The Son


In [40]:
# Test A — Zee5 WITH the vote gate (current behavior)
movies = discover_movies_by_genre(get_genre_id("Drama"), n=20, provider_ids=[232])
print("Zee5 Drama (with gate):", len(movies))

# Test B — Zee5 WITHOUT the vote gate
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "api_key": TMDB_API_KEY,
    "with_genres": get_genre_id("Drama"),
    "with_watch_providers": 232,
    "watch_region": "IN",
    "sort_by": "popularity.desc",
    # NO vote_count.gte
}
data = requests.get(url, params=params).json()
print("Zee5 Drama (no gate):", len(data.get("results", [])))
for m in data["results"][:8]:
    print(" ", m["title"], "| votes:", m.get("vote_count"))



Zee5 Drama (with gate): 20
Zee5 Drama (no gate): 20
  Oppenheimer | votes: 12208
  Top Gun: Maverick | votes: 11321
  Gladiator | votes: 21242
  Django Unchained | votes: 28373
  Schindler's List | votes: 17721
  Gladiator II | votes: 4642
  Shutter Island | votes: 26197
  The Brutalist | votes: 1735


In [41]:
movie_id = 361743  # or whatever movie you're checking
url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers"
data = requests.get(url, params={"api_key": TMDB_API_KEY}).json()
print(data["results"]["IN"])

{'link': 'https://www.themoviedb.org/movie/361743-top-gun-maverick/watch?locale=IN', 'flatrate': [{'logo_path': '/pvske1MyAoymrs5bguRfVqYiM9a.jpg', 'provider_id': 119, 'provider_name': 'Amazon Prime Video', 'display_priority': 1}, {'logo_path': '/kVqjgpcwvDJOhCupjcLzwwtOp52.jpg', 'provider_id': 2336, 'provider_name': 'JioHotstar', 'display_priority': 3}, {'logo_path': '/8aBqoNeGGr0oSA85iopgNZUOTOc.jpg', 'provider_id': 2100, 'provider_name': 'Amazon Prime Video with Ads', 'display_priority': 73}], 'rent': [{'logo_path': '/SPnB1qiCkYfirS2it3hZORwGVn.jpg', 'provider_id': 2, 'provider_name': 'Apple TV Store', 'display_priority': 5}, {'logo_path': '/gP67NRy1ShUJilrzMsbOmEmdmcv.jpg', 'provider_id': 232, 'provider_name': 'Zee5', 'display_priority': 7}, {'logo_path': '/8z7rC8uIDaTM91X0ZfkRf04ydj2.jpg', 'provider_id': 3, 'provider_name': 'Google Play Movies', 'display_priority': 8}, {'logo_path': '/pTnn5JwWr4p3pG8H6VrpiQo7Vs0.jpg', 'provider_id': 192, 'provider_name': 'YouTube', 'display_priori

In [42]:
movies = discover_movies_by_genre(get_genre_id("Drama"), n=20, provider_ids=[237])
count = 0
for m in movies:
    plats = get_providers_by_id(m["id"])
    if any("sony liv" in p.lower() for p in plats):
        count += 1
        print("✅", m["title"], "→", plats)
print("Sony Liv flatrate hits:", count, "/", len(movies))

✅ The Longest Yard → ['Sony Liv']
✅ Weathering with You → ['Crunchyroll', 'Sony Liv', 'Crunchyroll Amazon Channel']
✅ Seven Pounds → ['Sony Liv', 'Sony Pictures Amazon Channel']
✅ Big Fish → ['Sony Liv', 'VI movies and tv', 'Sony Pictures Amazon Channel']
✅ Captain Phillips → ['Amazon Prime Video', 'Sony Liv', 'Amazon Prime Video with Ads']
✅ Moneyball → ['Sony Liv', 'Sony Pictures Amazon Channel']
✅ Crouching Tiger, Hidden Dragon → ['Netflix', 'Amazon Prime Video', 'Sony Liv', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']
✅ Bicentennial Man → ['Netflix', 'Sony Liv']
✅ Triangle of Sadness → ['Sony Liv', 'VI movies and tv']
✅ Philadelphia → ['Sony Liv', 'Sony Pictures Amazon Channel']
✅ The Exorcism of Emily Rose → ['JioHotstar', 'Sony Liv']
✅ Searching → ['Sony Liv', 'VI movies and tv', 'Sony Pictures Amazon Channel']
✅ A League of Their Own → ['Sony Liv', 'VI movies and tv']
✅ As Good as It Gets → ['Sony Liv', 'Sony Pictures Amazon Channel']
✅ 5 Centimeters per Second

In [43]:
rejected, avoided = get_user_feedback(1)
print("REJECTED:", [r for r in rejected if "Boot" in r])

titles = get_recommendations(genre="Action", n=12, user_id=1)
print("IN TITLES:", [t for t in titles if "Boot" in t])

REJECTED: []
IN TITLES: []


In [44]:
rejected, avoided = get_user_feedback(1)
print("REJECTED titles:", rejected)

REJECTED titles: set()


In [45]:
print(get_user_feedback(1))


(set(), set())


In [46]:
# Test: does with_original_language actually narrow results?
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "api_key": TMDB_API_KEY,
    "with_genres": get_genre_id("Drama"),
    "with_original_language": "ko",   # Korean
    "sort_by": "popularity.desc",
}
data = requests.get(url, params=params).json()
for m in data.get("results", [])[:5]:
    print(m["title"], "|", m.get("original_language"))

Parasite | ko
Obsessed | ko
The Handmaiden | ko
Adultery Alumni Association 2 | ko
Oligosaccharide The Movie | ko


In [47]:
# Test 1: language + genre only, NO platform filter
intent = get_user_intent("I liked Vincenzo, suggest similar ones")
genre = resolve_genre(intent)
language = intent.get("language")
print("Genre:", genre, "| Language:", language)

results = guest_recommendations_with_platform(
    genre_name=genre, user_platforms=None, language=language, n=6)
print("Without platform filter:", len(results))
for m in results:
    print(" ", m["title"], "|", m.get("platforms"))

Genre: Drama | Language: Korean
Without platform filter: 2
  Cyber Hell: Exposing an Internet Horror | ['Netflix']
  Welcome to Dongmakgol | ['Channel K Amazon Channel']


In [48]:
for lang, code in [("Telugu", "te"), ("Kannada", "kn"), ("Malayalam", "ml")]:
    url = "https://api.themoviedb.org/3/discover/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "with_original_language": code,
        "sort_by": "popularity.desc",
        "vote_count.gte": 100,   # ← note this
    }
    data = requests.get(url, params=params).json()
    print(lang, "→", len(data.get("results", [])), "results")

Telugu → 17 results
Kannada → 4 results
Malayalam → 10 results


In [49]:
for lang, code in [("Telugu", "te"), ("Kannada", "kn"), ("Malayalam", "ml")]:
    for genre_name in ["Drama", "Comedy", "Crime"]:
        genre_id = get_genre_id(genre_name)
        params = {
            "api_key": TMDB_API_KEY,
            "with_original_language": code,
            "with_genres": genre_id,
            "sort_by": "popularity.desc",
            "vote_count.gte": 100,
        }
        data = requests.get(url, params=params).json()
        print(lang, "+", genre_name, "→", len(data.get("results", [])))

Telugu + Drama → 11
Telugu + Comedy → 2
Telugu + Crime → 3
Kannada + Drama → 1
Kannada + Comedy → 0
Kannada + Crime → 2
Malayalam + Drama → 6
Malayalam + Comedy → 5
Malayalam + Crime → 3


In [50]:
# Quick check: does dropping vote_count.gte help meaningfully?
for lang, code in [("Kannada", "kn")]:
    for genre_name in ["Comedy"]:
        genre_id = get_genre_id(genre_name)
        params = {
            "api_key": TMDB_API_KEY,
            "with_original_language": code,
            "with_genres": genre_id,
            "sort_by": "popularity.desc",
            # NO vote_count.gte
        }
        data = requests.get(url, params=params).json()
        print(len(data.get("results", [])))

20


In [51]:
# Test WITHOUT watch_region at all
params = {
    "api_key": TMDB_API_KEY,
    "with_original_language": "kn",
    "with_genres": get_genre_id("Comedy"),
    "sort_by": "popularity.desc",
}
data = requests.get(url, params=params).json()
print(len(data.get("results", [])))

20


In [52]:
print(data.get("total_results"), "total | showing", len(data.get("results", [])), "on this page")

415 total | showing 20 on this page


In [53]:
results = discover_movies_by_genre(get_genre_id("Comedy"), n=6, language="kn")
print(len(results))
for m in results:
    print(m["title"])

6
Bindaas
Vishnuvardhana
Mungaru Male
Dana Kayonu
Ganeshana Maduve
Hendtheer Darbar


In [54]:
# Check page-by-page: how many Kannada Comedy movies actually meet vote_count >= 20?
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "api_key": TMDB_API_KEY,
    "with_original_language": "kn",
    "with_genres": get_genre_id("Comedy"),
    "sort_by": "popularity.desc",
    "vote_count.gte": 20,
}
data = requests.get(url, params=params).json()
print("Total results (with vote gate):", data.get("total_results"))
print("Total pages:", data.get("total_pages"))

Total results (with vote gate): 7
Total pages: 1


In [55]:
# Test: does variety still work for the "Any" language / English path?
for _ in range(3):
    results = discover_movies_by_genre(get_genre_id("Comedy"), n=3)  # no language
    print([m["title"] for m in results])

['Pixels', 'Kingsman: The Secret Service', 'Bruce Almighty']
['Back to the Future', 'The Wrecking Crew', 'Big Hero 6']
['Back to the Future', 'The Wrecking Crew', 'Big Hero 6']


In [56]:
pages_seen = []
for _ in range(10):
    import random as r
    page = r.randint(1, 3)
    pages_seen.append(page)
print(pages_seen)

[3, 2, 3, 1, 3, 2, 3, 1, 2, 2]


In [57]:
for _ in range(4):
    r = discover_movies_by_genre(get_genre_id("Comedy"), n=3, language="kn")
    print([m["title"] for m in r])

['Chow Chow Bath', 'My Autograph', 'Karikaada']
['Simple Agi Ondh Love Story', 'The Devil', 'Romeo']
['Tribble Riding', 'Sri Ranga', 'Hero']
['Dakota Express', 'Khushi Khushiyagi', 'Mundina Nildana']
